In [1]:
import pandas as pd
import numpy as np

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:

def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [4]:
# df=pd.read_csv('elvisdenverco2024obweekday_export_odbc.csv')
df=pd.read_csv('elvisbartca2024interceptNEW_main_export_odbc(001).csv')
filename='details_project_od_BART.xlsx'
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,EXP_HOMELESSCode,EXP_HOMELESS,_REVERSE_TRIP,GENERATABLE_TRIPS
0,9,2024-04-08 06:09:51,83,en,2024-04-08 05:59:10,2024-06-06 03:30:02,174.249.144.185,https://bart-ca.etc-research.com/,-25200.0,999,...,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11,2024-04-08 07:16:13,83,en,2024-04-08 07:07:19,2024-04-08 07:16:13,107.77.213.232,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,14,2024-04-08 08:21:40,83,en,2024-04-08 07:16:22,2024-09-07 02:29:43,107.77.213.232,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,999,...,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,15,2024-04-08 07:57:27,83,en,2024-04-08 07:44:11,2024-07-04 11:40:55,172.58.89.249,https://bart-ca.etc-research.com/,-25200.0,JWR,...,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16,2024-04-08 08:00:14,83,en,2024-04-08 07:49:17,2024-07-05 11:33:04,174.249.144.185,https://bart-ca.etc-research.com/,-25200.0,EJG,...,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df['Completed']=pd.to_datetime(df['Completed'], errors='coerce')
df['Survey Date'] = df['Completed'].dt.date

In [6]:
df[['Survey Date']]

,Survey Date
0,2024-04-08
1,2024-04-08
2,2024-04-08
3,2024-04-08
4,2024-04-08
...,...
2982,2024-09-05
2983,2024-09-05
2984,2024-09-05
2985,2024-09-05


In [7]:
df=df[(df['INTERV_INIT']!='999')]
df = df[(df['HAVE_5_MIN_FOR_SURVECode'] == 1)]

In [8]:
def details_dataframe(filename):
    details_df=pd.read_excel(filename,sheet_name='AIR')
    return details_df


home_airport_hotel_column_names=['originaddresslat','originaddresslong', 'destinaddresslat', 'destinaddresslong','originplacetype','homeaddresslat','homeaddresslong',
                                 'destinairportcode','destinplacetype']

def check_home_airport_hotel(df, details_df):

    # Loop through each row in the DataFrame 'df'

    data_list = check_all_characters_present(df,home_airport_hotel_column_names)
    data_list.sort()
    
    for _, row in df.iterrows():
        # Extract latitude and longitude values for origin and destination
        
        origin_addr_lat = row[data_list[6]]
        origin_addr_lng = row[data_list[7]]
        destin_addr_lat = row[data_list[0]]
        destin_addr_lng = row[data_list[1]]

        # Check if origin latitude and longitude are missing
        if pd.isna(origin_addr_lat) or pd.isna(origin_addr_lng):
            print("Origin Lat Long Not present")
            # Get the type of origin place
            origin_place_type = row['ORIGIN_PLACE_TYPE']
            lat_col, lng_col = None, None  # Initialize variables for latitude and longitude column names
            if not pd.isna(origin_place_type):
            # Determine the appropriate columns based on place type
                if 'hotel' in origin_place_type.lower() or 'home' in origin_place_type.lower():
                    lat_col, lng_col = 'HOME_ADDRESS_LAT','HOME_ADDRESS_LONG'
                    print('origin placetype is home')
                elif 'airport' in origin_place_type.lower():
                    airport_destin_code = row['DESTIN_AIRPORTCode']
                    airport_row = details_df[details_df['LIME_CODE'] == airport_destin_code]
                    if not airport_row.empty:
                        lat_col, lng_col = 'lat6', 'lng6'

                # If valid latitude and longitude columns are found, populate the corresponding values
                if lat_col and lng_col:
                    print(f'values for home lat long {row[lat_col]} {row[lng_col]}')
                    df.at[row.name, 'ORIGIN_ADDRESS_LAT'] = row[lat_col]
                    df.at[row.name, 'ORIGIN_ADDRESS_LONG'] = row[lng_col]

        # Check if destination latitude and longitude are missing
        if pd.isna(destin_addr_lat) or pd.isna(destin_addr_lng):
            # Get the type of destination place
            print("Destin Lat Long Not Present")
            destin_place_type = row['DESTIN_PLACE_TYPE']
            print('destin placetype is',destin_place_type)
            lat_col, lng_col = None, None  # Initialize variables for latitude and longitude column names
            if not pd.isna(destin_place_type):
#                 print("Destination place type is NaN, skipping this row")
#                 continue
            # Determine the appropriate columns based on place type
                if 'hotel' in destin_place_type.lower() or 'home' in destin_place_type.lower():
                    print('destin placetype is home/hotel')
                    lat_col, lng_col = 'HOME_ADDRESS_LAT','HOME_ADDRESS_LONG'
                elif 'airport' in destin_place_type.lower():
                    airport_destin_code = row['DESTIN_AIRPORTCode']
                    airport_row = details_df[details_df['LIME_CODE'] == airport_destin_code]
                    if not airport_row.empty:
                        lat_col, lng_col = 'lat6', 'lng6'

                # If valid latitude and longitude columns are found, populate the corresponding values
                if lat_col and lng_col:
                    print(f'values for home lat long {row[lat_col]} {row[lng_col]}')
                    df.at[row.name, 'DESTIN_ADDRESS_LAT'] = row[lat_col]
                    df.at[row.name, 'DESTIN_ADDRESS_LONG'] = row[lng_col]
    # Return the DataFrame 'df' with the populated columns
    return df


In [9]:
origin_destin_columns_check=['originaddresslat','originaddresslong','destinaddresslat','destinaddresslong']
destin_columns_check=['destinaddresslat','destinaddresslong']
origin_columns_check=['originaddresslat','originaddresslong']
origin_destin_columns=check_all_characters_present(df,origin_destin_columns_check)
origin_columns=check_all_characters_present(df,origin_columns_check)
destin_columns=check_all_characters_present(df,destin_columns_check)
origin_destin_columns,origin_columns,destin_columns

(['ORIGIN_ADDRESS_LAT',
  'ORIGIN_ADDRESS_LONG',
  'DESTIN_ADDRESS_LAT',
  'DESTIN_ADDRESS_LONG'],
 ['ORIGIN_ADDRESS_LAT', 'ORIGIN_ADDRESS_LONG'],
 ['DESTIN_ADDRESS_LAT', 'DESTIN_ADDRESS_LONG'])

In [10]:
new_df=df[(df['id']==2650)|(df['id']==5472)|(df['id']==5547)|(df['id']==5562)|(df['id']==5570)][['id','ORIGIN_PLACE_TYPE',*origin_destin_columns,'DESTIN_PLACE_TYPE', 'HOME_ADDRESS_LAT','HOME_ADDRESS_LONG']]

In [11]:
details_df=details_dataframe(filename)

df=check_home_airport_hotel(df,details_df)

Origin Lat Long Not present
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.569729 -122.331034
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.548307 -122.309956
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.61048 -122.395857
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.96234 -122.010053
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.5875 -122.371903
Origin Lat Long Not present
origin placetype is home
values for home lat long 38.018285 -121.924555
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 38.018285 -121.924555
Origin Lat Long Not present
origin placetype is home
values for home lat long 38.005638 -1

Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.790839 -122.412769
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.778922 -122.423706
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.858712 -122.280262
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.706096 -122.121356
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.956672 -122.348463
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.764009 -122.418489
Destin Lat Long Not Present
destin placetype is Home
destin placetype is home/hotel
values for home lat long 37.942872 -122.346576
Origin Lat Long Not present
origin placetype is home
values for home lat long 37.87

In [12]:
new_df

,id,ORIGIN_PLACE_TYPE,ORIGIN_ADDRESS_LAT,ORIGIN_ADDRESS_LONG,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG,DESTIN_PLACE_TYPE,HOME_ADDRESS_LAT,HOME_ADDRESS_LONG
842,2650,Home,NaN,NaN,37.794031,-122.394941,Workplace,37.980347,-122.031224
2548,5472,Workplace,37.698496,-121.926824,NaN,NaN,Home,37.791304,-122.407369
2612,5547,Workplace,37.697803,-121.925466,NaN,NaN,Home,37.800367,-122.216156
2623,5562,Workplace,37.697803,-121.925466,NaN,NaN,Home,37.796564,-122.256462
2631,5570,Workplace,37.697803,-121.925466,NaN,NaN,Home,37.795128,-122.276470


In [13]:
df[(df['id']==2650)|(df['id']==5472)|(df['id']==5547)|(df['id']==5562)|(df['id']==5570)][['id','ORIGIN_PLACE_TYPE',*origin_destin_columns,'DESTIN_PLACE_TYPE', 'HOME_ADDRESS_LAT','HOME_ADDRESS_LONG']]

,id,ORIGIN_PLACE_TYPE,ORIGIN_ADDRESS_LAT,ORIGIN_ADDRESS_LONG,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG,DESTIN_PLACE_TYPE,HOME_ADDRESS_LAT,HOME_ADDRESS_LONG
842,2650,Home,37.980347,-122.031224,37.794031,-122.394941,Workplace,37.980347,-122.031224
2548,5472,Workplace,37.698496,-121.926824,37.791304,-122.407369,Home,37.791304,-122.407369
2612,5547,Workplace,37.697803,-121.925466,37.800367,-122.216156,Home,37.800367,-122.216156
2623,5562,Workplace,37.697803,-121.925466,37.796564,-122.256462,Home,37.796564,-122.256462
2631,5570,Workplace,37.697803,-121.925466,37.795128,-122.276470,Home,37.795128,-122.276470


In [14]:
df.columns

Index(['id', 'Completed', 'Last_page', 'Start_language', 'Date_started',
       'Date_last_action', 'IP_address', 'Referring_URL', 'TIME_ADJUST',
       'INTERV_INIT',
       ...
       'INVITE_EMAIL', 'INVITE_SMS', 'INVITE_CALL', 'INVITE_TOKEN',
       'INVITE_STATUS', 'EXP_HOMELESSCode', 'EXP_HOMELESS', '_REVERSE_TRIP',
       'GENERATABLE_TRIPS', 'Survey Date'],
      dtype='object', length=862)

In [15]:
df.drop_duplicates(subset=origin_destin_columns)

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,EXP_HOMELESSCode,EXP_HOMELESS,_REVERSE_TRIP,GENERATABLE_TRIPS,Survey Date
3,15,2024-04-08 07:57:27,83,en,2024-04-08 07:44:11,2024-07-04 11:40:55,172.58.89.249,https://bart-ca.etc-research.com/,-25200.0,JWR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-08
4,16,2024-04-08 08:00:14,83,en,2024-04-08 07:49:17,2024-07-05 11:33:04,174.249.144.185,https://bart-ca.etc-research.com/,-25200.0,EJG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-08
5,18,2024-04-08 08:10:24,83,en,2024-04-08 08:02:52,2024-07-05 11:32:46,174.249.147.91,NaN,-25200.0,MOW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-08
6,19,2024-04-08 08:34:30,83,en,2024-04-08 08:03:10,2024-07-05 11:29:16,172.58.89.249,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,JWR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-08
7,20,2024-04-08 08:15:17,83,en,2024-04-08 08:08:34,2024-07-04 12:16:31,174.249.144.185,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,EJG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2978,7080,2024-09-05 20:29:12,86,en,2024-09-05 19:17:05,2024-09-09 08:20:31,172.59.131.89,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,CVD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-09-05
2979,7082,2024-09-05 19:24:21,86,en,2024-09-05 19:17:44,2024-09-09 08:20:52,107.77.214.51,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,EVE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-09-05
2981,7084,2024-09-05 20:30:11,86,en,2024-09-05 19:27:37,2024-09-09 08:21:04,166.216.158.194,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,SAR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-09-05
2983,7086,2024-09-05 20:22:58,86,en,2024-09-05 20:17:58,2024-09-09 08:21:20,107.77.214.51,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,EVE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-09-05


In [16]:
duplicated_df = df[df.duplicated(subset=origin_destin_columns, keep=False)]
duplicated_df

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,EXP_HOMELESSCode,EXP_HOMELESS,_REVERSE_TRIP,GENERATABLE_TRIPS,Survey Date
340,1605,2024-05-15 18:00:05,86,en,2024-05-15 17:54:54,2024-06-18 22:16:14,166.216.158.76,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,MDE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-15
342,1610,2024-05-15 18:04:24,86,en,2024-05-15 18:00:47,2024-05-15 18:04:24,166.216.158.76,https://bart-ca.etc-research.com/,-25200.0,MDE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-15
740,2488,2024-05-23 07:24:16,86,en,2024-05-23 07:13:25,2024-05-23 07:24:16,107.77.213.217,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,SAW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-23
946,2837,2024-05-28 11:24:53,86,en,2024-05-28 11:11:04,2024-05-28 11:24:53,107.77.212.190,NaN,-25200.0,MDE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-28
964,2862,2024-05-28 12:25:35,86,en,2024-05-28 12:18:26,2024-05-28 12:25:35,107.77.214.34,NaN,-25200.0,JMR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-28
972,2880,2024-05-28 13:25:46,86,en,2024-05-28 13:18:20,2024-05-28 13:25:46,174.249.147.30,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,ZLY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-28
976,2889,2024-05-28 13:48:22,86,en,2024-05-28 13:26:16,2024-05-28 13:48:22,174.249.147.30,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,ZLY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-28
985,2902,2024-05-28 13:52:35,86,en,2024-05-28 13:48:43,2024-05-28 13:52:35,174.249.147.30,https://bart-ca.etc-research.com/index.php/952...,-25200.0,ZLY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-28
990,2913,2024-05-28 14:13:28,86,en,2024-05-28 14:04:14,2024-05-28 14:13:28,174.249.147.30,https://bart-ca.etc-research.com/index.php/sur...,-25200.0,ZLY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-28
995,2919,2024-05-28 14:23:50,86,en,2024-05-28 14:13:50,2024-05-28 14:23:50,174.249.147.30,https://bart-ca.etc-research.com/index.php/952...,-25200.0,ZLY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-28


In [17]:
unique_intervs=df['INTERV_INIT'].unique()
unique_intervs

array(['JWR', 'EJG', 'MOW', 'ICF', 'ANA', 'DNP', 'HRD', 'JAC', 'ODA',
       'ASS', 'LIZ', 'AJC', 'SAW', 'URY', 'CSO', 'P.R', 'KJT', 'AEB',
       'JFP', 'ALM', 'MDE', 'MD3', 'HHK', 'BDM', 'LMM', 'DDF', 'RAS',
       'JLR', 'MCM', 'KHC', 'SJH', 'BWC', 'BPB', 'ZLY', 'ANS', 'JMR',
       'EVE', 'ISA', 'CBA', 'CMH', 'RMM', 'LZZ', 'AMT', 'KMR', 'JMB',
       'AEW', 'JTT', 'JAT', 'CIS', 'HPF', 'RCM', 'STR', 'DJS', 'ALD',
       'WJJ', 'CS0', 'KRD', 'CVD', 'BEA', 'SAR', 'TAX'], dtype=object)

In [18]:
duplicated_df[duplicated_df['INTERV_INIT']=='ZLY'][origin_destin_columns]
# duplicated_df[duplicated_df['INTERV_INIT']=='ZLY'][['id','INTERV_INIT',*origin_destin_columns]].to_csv('INTERVIEWER_Duplicated_data01.csv',index=False)

,ORIGIN_ADDRESS_LAT,ORIGIN_ADDRESS_LONG,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG
972,37.897808,-122.064147,37.852157,-122.286606
976,37.897808,-122.064147,37.852157,-122.286606
985,37.897808,-122.064147,37.852157,-122.286606
990,37.897808,-122.064147,37.852157,-122.286606
995,37.897808,-122.064147,37.852157,-122.286606


In [19]:
for interv in unique_intervs:
    if interv=='ZLY':
        user_data=df[df['INTERV_INIT']==interv]
        user_duplicated_df=user_data[user_data.duplicated(subset=origin_destin_columns, keep=False)]
        user_duplicated_df[['id','INTERV_INIT',*origin_destin_columns]].to_csv('INTERVIEWER_Duplicated_data02.csv',index=False)

In [20]:
df[(df['id']==5472)|(df['id']==5547)|(df['id']==5562)|(df['id']==5570)][['id',*origin_destin_columns]]

,id,ORIGIN_ADDRESS_LAT,ORIGIN_ADDRESS_LONG,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG
2548,5472,37.698496,-121.926824,37.791304,-122.407369
2612,5547,37.697803,-121.925466,37.800367,-122.216156
2623,5562,37.697803,-121.925466,37.796564,-122.256462
2631,5570,37.697803,-121.925466,37.795128,-122.276470


In [21]:
final_duplicated_df = pd.DataFrame()
final_destin_duplicated_df = pd.DataFrame()
final_origin_duplicated_df = pd.DataFrame()

for interv in unique_intervs:
    
    user_data = df[df['INTERV_INIT'] == interv]
    unique_dates = user_data['Survey Date'].unique()

    for date in unique_dates:

        date_data = user_data[user_data['Survey Date'] == date]

        # Check for duplicated rows based on origin, destination, and both combined
        user_origin_duplicated_df = date_data[date_data.duplicated(subset=origin_columns, keep=False)]
        user_destin_duplicated_df = date_data[date_data.duplicated(subset=destin_columns, keep=False)]
        user_duplicated_df = date_data[date_data.duplicated(subset=origin_destin_columns, keep=False)]

        # Append results to the final DataFrames
        final_origin_duplicated_df = pd.concat([
            final_origin_duplicated_df, 
            user_origin_duplicated_df[['id', 'Survey Date', 'INTERV_INIT', 'ORIGIN_PLACE_TYPE', *origin_columns, 'HOME_ADDRESS_LAT', 'HOME_ADDRESS_LONG']]
        ])

        final_destin_duplicated_df = pd.concat([
            final_destin_duplicated_df, 
            user_destin_duplicated_df[['id', 'Survey Date', 'INTERV_INIT', *destin_columns, 'DESTIN_PLACE_TYPE', 'HOME_ADDRESS_LAT', 'HOME_ADDRESS_LONG']]
        ])

        final_duplicated_df = pd.concat([
            final_duplicated_df, 
            user_duplicated_df[['id', 'Survey Date', 'INTERV_INIT', 'ORIGIN_PLACE_TYPE', *origin_destin_columns, 'DESTIN_PLACE_TYPE', 'HOME_ADDRESS_LAT', 'HOME_ADDRESS_LONG']]
        ])

# Reset index for all DataFrames after concatenation
final_origin_duplicated_df.reset_index(drop=True, inplace=True)
final_destin_duplicated_df.reset_index(drop=True, inplace=True)
final_duplicated_df.reset_index(drop=True, inplace=True)

In [22]:
# final_duplicated_df.to_csv('Final_Interviewer_Data001.csv',index=False)

In [23]:
with pd.ExcelWriter(f'INTERV_Duplicated_Records.xlsx') as writer:
    # grouped[['INTERV_INIT','Description']].to_excel(writer, sheet_name="INTERV One Line Summary", index=False)
    final_origin_duplicated_df.to_excel(writer, sheet_name="INTERV Origin Duplicates", index=False)    
    final_destin_duplicated_df.to_excel(writer, sheet_name="INTERV Destin Duplicates", index=False)    
    final_duplicated_df.to_excel(writer, sheet_name="INTERV Overall Duplicates", index=False)

In [24]:
final_duplicated_df = pd.DataFrame()
final_destin_duplicated_df = pd.DataFrame()
final_origin_duplicated_df = pd.DataFrame()
origin_data=[]
destin_data=[]
origin_destin_data=[]
for interv in unique_intervs:
    
    user_data = df[df['INTERV_INIT'] == interv]
    unique_dates = user_data['Survey Date'].unique()
    for date in unique_dates:
        
        date_data = user_data[user_data['Survey Date'] == date]

        # Check for duplicated rows based on origin, destination, and both combined
        user_origin_duplicated_df = date_data[date_data.duplicated(subset=origin_columns, keep=False)]
        user_destin_duplicated_df = date_data[date_data.duplicated(subset=destin_columns, keep=False)]
        user_duplicated_df = date_data[date_data.duplicated(subset=origin_destin_columns, keep=False)]
        
        total_records=date_data.shape[0]
        origin_flag_records=user_origin_duplicated_df.shape[0]
        origin_flag_records_ids=user_origin_duplicated_df['id'].values
        destin_flag_records=user_destin_duplicated_df.shape[0]
        destin_flag_records_ids=user_destin_duplicated_df['id'].values
        origin_destin_flag_records=user_duplicated_df.shape[0]
        origin_destin_flag_records_ids=user_duplicated_df['id'].values
        
        # Append results to the final DataFrames
        final_origin_duplicated_df = pd.concat([
            final_origin_duplicated_df, 
            user_origin_duplicated_df[['id', 'Survey Date', 'INTERV_INIT', 'ORIGIN_PLACE_TYPE', *origin_columns, 'HOME_ADDRESS_LAT', 'HOME_ADDRESS_LONG']]
        ])

        final_destin_duplicated_df = pd.concat([
            final_destin_duplicated_df, 
            user_destin_duplicated_df[['id', 'Survey Date', 'INTERV_INIT', *destin_columns, 'DESTIN_PLACE_TYPE', 'HOME_ADDRESS_LAT', 'HOME_ADDRESS_LONG']]
        ])

        final_duplicated_df = pd.concat([
            final_duplicated_df, 
            user_duplicated_df[['id', 'Survey Date', 'INTERV_INIT', 'ORIGIN_PLACE_TYPE', *origin_destin_columns, 'DESTIN_PLACE_TYPE', 'HOME_ADDRESS_LAT', 'HOME_ADDRESS_LONG']]
        ])
        origin_data.append({
            'INTERV INIT':interv,
            'Survey Date':date,
            'Origin Duplicated Records':origin_flag_records,
            'Total Records':total_records,
            'Origin Duplicated Percentage':round(origin_flag_records/total_records*100,2) if total_records > 0 else 0,
            "Origin Flags IDS":origin_flag_records_ids
            
        })
        destin_data.append({
            'INTERV INIT':interv,
            'Survey Date':date,
            'Origin Duplicated Records':destin_flag_records,
            'Total Records':total_records,
            'Origin Duplicated Percentage':round(destin_flag_records/total_records*100,2) if total_records > 0 else 0,
            "Origin Flags IDS":destin_flag_records_ids
            
        })
        origin_destin_data.append({
                'INTERV INIT':interv,
                'Survey Date':date,
                'Origin Destin Duplicated Records':origin_destin_flag_records,
                'Total Records':total_records,
                'Origin Destin Duplicated Percentage':round(origin_destin_flag_records/total_records*100,2) if total_records > 0 else 0,
                "Origin Destin Flags IDS":origin_destin_flag_records_ids

            })

# Reset index for all DataFrames after concatenation
final_origin_duplicated_df.reset_index(drop=True, inplace=True)
final_destin_duplicated_df.reset_index(drop=True, inplace=True)
final_duplicated_df.reset_index(drop=True, inplace=True)
origin_df=pd.DataFrame(origin_data)
destin_df=pd.DataFrame(destin_data)
origin_destin_df=pd.DataFrame(origin_destin_data)

In [25]:
with pd.ExcelWriter(f'INTERV_Duplicated_stats03.xlsx') as writer:
    # grouped[['INTERV_INIT','Description']].to_excel(writer, sheet_name="INTERV One Line Summary", index=False)
    origin_df.to_excel(writer, sheet_name="INTERV Daily Origin%", index=False)    
    destin_df.to_excel(writer, sheet_name="INTERV Daily Destin%", index=False)    
    origin_destin_df.to_excel(writer, sheet_name="INTERV Daily OriginDestin%", index=False)    
    final_origin_duplicated_df.to_excel(writer, sheet_name="INTERV Origin Duplicates", index=False)    
    final_destin_duplicated_df.to_excel(writer, sheet_name="INTERV Destin Duplicates", index=False)    
    final_duplicated_df.to_excel(writer, sheet_name="INTERV Overall Duplicates", index=False)